# 📘 Colab Notebook: Evaluate LLM Responses from SQL Databases (PostgreSQL/MySQL) Using Llumo

## 📝 Notebook Overview
This notebook demonstrates how to load your existing LLM interaction logs from a SQL database (PostgreSQL or MySQL), format the data, and then evaluate the model's responses using Llumo’s powerful input-level metrics to ensure quality and safety.

### ✨ Metrics included:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
- 🛠️ Context Utilization
  
---

## 🚀 What you will do in this notebook:
- 🗄️ Connect to your SQL database and load data from a table.  
- 🔄 Format the raw data from SQL rows into the standardized structure required by Llumo.
- 🤖 Evaluate the model's output for correctness, completeness, bias, harmfulness, and more.
- 📊 View the detailed evaluation results in a structured table.  
---

### **⚙️ 1. Install Dependencies**
First, we'll install the necessary Python libraries:
- `llumo`: The official SDK for the Llumo platform.
- `pandas`: Used for data manipulation.
- `SQLAlchemy`: A toolkit that provides a common interface for different SQL databases.
- `psycopg2-binary`: The database driver for PostgreSQL.
- `mysql-connector-python`: The database driver for MySQL.

In [ ]:
!pip install llumo pandas sqlalchemy psycopg2-binary mysql-connector-python -q

### **📚 2. Import Required Libraries**

In [ ]:
import os
import pandas as pd
import json
import getpass
from llumo import LlumoClient
from sqlalchemy import create_engine, text

### **🔑 3. Configure API Key & Database Connection Details**

To use Llumo and access your database, you need to set up the appropriate credentials. 

1.  **Llumo API Key**: Get your key from the [Llumo Dashboard](https://llumo.ai/dashboard).
2.  **Database Credentials**: Provide the connection details for your PostgreSQL or MySQL instance.

In [ ]:
# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter Your Llumo API Key: ")
llumo_key = os.getenv("LLUMO_API_KEY")

# ⚙️ Set your Database connection details
db_type = input("Enter database type ('postgresql' or 'mysql'): ").lower().strip()
db_user = input("Enter database user: ")
db_password = getpass.getpass("Enter database password: ")
db_host = input("Enter database host (e.g., 'localhost' or an IP address): ")
db_port = input(f"Enter database port (e.g., 5432 for Postgres, 3306 for MySQL): ")
db_name = input("Enter database name: ")
table_name = input("Enter the name of the table with your logs: ")

# Construct the database connection URI for SQLAlchemy
engine_uri = None
if db_type == 'postgresql':
    engine_uri = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
elif db_type == 'mysql':
    engine_uri = f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
else:
    print("Invalid database type entered. Please run the cell again and choose 'postgresql' or 'mysql'.")

### **🗄️ 4. Read Data from SQL Database**

This step uses the credentials you provided to connect to the database and fetch data from your specified table. We use `pandas` and `SQLAlchemy` to execute a simple `SELECT` query and load the results directly into a DataFrame.

**Note**: The query is limited to the first 100 rows for this example. You can adjust or remove the `LIMIT 100` clause in the SQL query to fetch more data.

In [ ]:
raw_logs = []

if engine_uri:
    try:
        # Create a database engine
        print("Connecting to the database...")
        engine = create_engine(engine_uri)
        
        # Define the SQL query to fetch data
        # Using text() from SQLAlchemy to prevent SQL injection risks with table names
        query = text(f'SELECT * FROM "{table_name}" LIMIT 100')
        
        # Execute the query and load data into a pandas DataFrame
        with engine.connect() as connection:
            print(f"Fetching data from table '{table_name}'...")
            raw_df = pd.read_sql_query(query, connection)
            # Convert the DataFrame to a list of dictionaries
            raw_logs = raw_df.to_dict(orient='records')
        
        print(f"Successfully loaded {len(raw_logs)} records from the database.")

    except Exception as e:
        print(f"An error occurred while connecting or fetching data: {e}")
        print("Please check your database credentials, host, and table name.")

# Preview the first raw log to understand its structure
if raw_logs:
    print("\nSample raw log from the database:")
    print(json.dumps(raw_logs[0], indent=2))

### **🔄 5. Format Data for Llumo Evaluation**
Llumo's `evaluateMultiple` function expects a list of dictionaries, where each dictionary contains specific keys like `query`, `context`, and `output`. 

The function below converts our raw data (from SQL rows) into this standardized format. **You must adjust the key mappings** inside the function to match the column names in your database table.

In [ ]:
def convert_to_llumo_format(logs):
  """
  Converts a list of raw log dictionaries from a SQL query into the format required by Llumo.
  
  Args:
    logs (list): A list of dictionaries, where each dictionary represents a row from the SQL table.
    
  Returns:
    list: A list of formatted dictionaries for Llumo evaluation.
  """
  formatted_data = []
  for log in logs:
    # ➡️ TODO: Adjust these key names to match your table's column names.
    # For example, if your user's prompt is in a column named 'prompt',
    # change 'user_query' to 'prompt'.
    formatted_dict = {
        'query': log.get('user_query', ''),           # Map your column for the user's question/prompt
        'context': log.get('retrieved_context', ''),  # Map your column for the retrieved context
        'output': log.get('model_answer', ''),        # Map your column for the model's generated response
        # Optional: Map the ground truth column if you have one
        'ground_truth': log.get('reference_answer', None) 
    }
    formatted_data.append(formatted_dict)
  return formatted_data

# Process the loaded logs
if raw_logs:
    evaluation_data = convert_to_llumo_format(raw_logs)
    
    # Preview the first formatted item to verify the mapping
    print("Sample log after formatting for Llumo:")
    print(json.dumps(evaluation_data[0], indent=2))
else:
    evaluation_data = []
    print("No data to format.")

### **🤖 6. Initialize Llumo Client and Evaluate Responses**

With our data loaded and formatted, we can now proceed with the evaluation. We will initialize the `LlumoClient` and call the `evaluateMultiple` function.

We pass our formatted data and select the KPIs we want to measure:
- 🎯 **Response Correctness**: Is the answer factually accurate based on the context?
- 🧩 **Response Completeness**: Does the answer fully address the user's query?
- 🧠 **Response Bias**: Is the response free from demographic or social biases?
- ☣️ **Harmfulness**: Does the response contain toxic, hateful, or unsafe content?
- 🛠️ **Context Utilization**: How well does the answer use the provided context?
- ▶ **Hallucination**: Does the answer invent information not present in the context?

In [ ]:
evalDf = pd.DataFrame()

if evaluation_data and llumo_key:
    # Initialize the LlumoClient with your API key
    client = LlumoClient(api_key = llumo_key)

    # Call the evaluation function
    print("Starting evaluation with Llumo...")
    evalDf = client.evaluateMultiple(
      data = evaluation_data,  # The formatted data from the previous step
      evals = ["Response Completeness", "Response Correctness", "Response Bias", "Context Utilization", "Hallucination"], # Selected evaluation KPIs
      getDataFrame = True # Return result as a pandas DataFrame
    )
    print("Evaluation complete!")
else:
    print("Skipping evaluation. Ensure data was loaded from the database and the Llumo API key is set.")

### **📊 7. View Evaluation Results**
The results are returned in a pandas DataFrame, providing a detailed breakdown of each metric for every row of data. This allows for easy analysis, sorting, and filtering to identify problematic responses and gain insights into your model's performance.

In [ ]:
# Display the full evaluation results table
if not evalDf.empty:
    # Configure pandas to display wide columns for better readability
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 80)
    display(evalDf)
else:
    print("No evaluation results to display.")